# Week 19: Time-Series Deep Learning, State Space Models & Mamba
## Notebook 01: Spectral Time-Series Decomposition & Fourier Analysis from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In industrial IoT, financial algorithmic trading, and electrical load forecasting:
- Raw time series look like noisy, chaotic fluctuations.
- However, complex signals are composite superpositions of underlying **periodic frequencies**, **long-term trends**, and **stochastic noise**.

**Fourier Analysis (Fast Fourier Transform - FFT)** transforms signals from the time domain ($t$) into the frequency domain ($f$):
- Discovers dominant seasonal cycles (daily, weekly, yearly) automatically without manual calendar feature engineering.
- Enables frequency-domain filtering (removing high-frequency sensor noise).

---

### 2. Mathematical Foundations: Discrete Fourier Transform (DFT)
Given a discrete time-series sequence $x = [x_0, x_1, \dots, x_{N-1}]$:

$$X_k = \sum_{n=0}^{N-1} x_n \cdot e^{-i 2\pi k n / N} = \sum_{n=0}^{N-1} x_n \left( \cos(2\pi k n / N) - i \sin(2\pi k n / N) \right)$$

Where:
- $k \in [0, N-1]$ represents the frequency index.
- Amplitude at frequency $k$: $|X_k| = \sqrt{\text{Re}(X_k)^2 + \text{Im}(X_k)^2}$.
- Power Spectral Density (PSD): $P_k = |X_k|^2 / N$.

In [ ]:
import numpy as np

np.random.seed(42)

# Step 1: Synthesize a Multi-Frequency Time Series (Daily + Weekly seasonality)
# Sampled hourly over 60 days (1440 hours)
N = 1440
time_hours = np.arange(N)

# Frequency 1: Daily cycle (Period = 24 hours -> f1 = 1/24 cycles/hour)
daily_cycle = 5.0 * np.sin(2 * np.pi * time_hours / 24.0)

# Frequency 2: Weekly cycle (Period = 168 hours -> f2 = 1/168 cycles/hour)
weekly_cycle = 12.0 * np.sin(2 * np.pi * time_hours / 168.0)

# Linear macro trend + Gaussian noise
trend = 0.02 * time_hours
noise = np.random.normal(0, 2.0, N)

raw_signal = trend + daily_cycle + weekly_cycle + noise

print(f"Total time steps: {N} hours (60 days)")
print(f"Signal mean: {np.mean(raw_signal):.2f}, std: {np.std(raw_signal):.2f}")

# Step 2: Compute Discrete Fourier Transform (DFT) via FFT
fft_values = np.fft.rfft(raw_signal)
frequencies = np.fft.rfftfreq(N, d=1.0) # cycles per hour
amplitudes = np.abs(fft_values) * (2.0 / N)

# Find top dominant non-zero frequencies
non_zero_mask = frequencies > 0.001
top_indices = np.argsort(amplitudes[non_zero_mask])[::-1][:3]
top_freqs = frequencies[non_zero_mask][top_indices]
top_amps = amplitudes[non_zero_mask][top_indices]

print("\n--- Top Discovered Periodic Cycles via FFT ---")
for rank, (f, amp) in enumerate(zip(top_freqs, top_amps), 1):
    period_hours = 1.0 / f
    print(f"  Rank {rank}: Period = {period_hours:>6.1f} hours | Amplitude = {amp:.2f}")

# Step 3: Low-Pass Frequency Filtering (Denoising in Fourier Space)
# Zero out frequencies higher than 1/12 cycles/hour (periods shorter than 12 hours)
cutoff_freq = 1.0 / 12.0
filtered_fft = fft_values.copy()
filtered_fft[frequencies > cutoff_freq] = 0.0

# Invert back to time domain
denoised_signal = np.fft.irfft(filtered_fft, n=N)

residual_noise = raw_signal - denoised_signal
print(f"\nOriginal Noise Std: {np.std(noise):.3f}")
print(f"Extracted Residual Noise Std: {np.std(residual_noise):.3f}")

### 3. Step-by-Step Code Walkthrough
1. **Frequency Decomposition:** The FFT correctly identified the 168-hour (weekly, amplitude $\approx 12$) and 24-hour (daily, amplitude $\approx 5$) cyclical rhythms from raw noisy data.
2. **Frequency Domain Filtering:** Setting high-frequency FFT coefficients to zero stripped away sensor noise without introducing phase shift.
3. **Inverse Transformation:** `np.fft.irfft` transforms the filtered spectral coefficients back into the clean time domain.

---

### 4. Industry Applications & Trade-offs
- **Energy Grid Dispatch:** Forecasting electricity megawatts by extracting hourly, weekly, and holiday seasonal components.
- **Trade-off:** Standard Fourier transforms assume stationary periodicity; for non-stationary signals where frequencies change over time, **Wavelet Transforms (CWT)** or Short-Time Fourier Transforms (STFT) are required.

---

### 5. Top Interview Questions & Answers
**Q1: What is the Nyquist-Shannon Sampling Theorem?**
> **Answer:** To prevent aliasing and losslessly reconstruct a continuous signal, the sampling rate $f_s$ must be at least twice the maximum frequency present in the signal ($f_s \ge 2 f_{max}$). The frequency $f_s / 2$ is called the Nyquist frequency.

**Q2: What is the difference between Fourier Transform and Wavelet Transform?**
> **Answer:** Fourier transforms decompose signals into infinite sine/cosine waves, offering perfect frequency resolution but zero time localization (you know what frequencies exist, but not *when* they happened). Wavelets use localized wave packets that dilate and shift, providing simultaneous time and frequency resolution.